Кр 1 Пикалова Ольга

In [11]:
!mkdir -p data
!wget https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta.txt.zip -P data
!unzip -o data/lenta.txt.zip -d data/

zsh:1: command not found: wget
Archive:  data/lenta.txt.zip
  inflating: data//lenta.txt         
  inflating: data//__MACOSX/._lenta.txt  


In [ ]:
L = r'[^\W\d_]'
pattern = re.compile(rf'(?<!{L})(?={L}*[а-яё])(?={L}*[a-z]){L}+', re.I)

found = pattern.findall(text)             # исходный текст, без предобработки
cnt = Counter(found)
print('всего вхождений:', len(found), '| уникальных:', len(cnt))
print('примеры:', list(cnt)[:10])
print('\nтоп-10 (регистр как в тексте):', cnt.most_common(10))

cnt_low = Counter(w.lower() for w in found)
print('топ-5 без учёта регистра:', cnt_low.most_common(5))

всего вхождений: 413 | уникальных: 282
примеры: ['машинs', 'Cледующим', 'cсылкой', 'Cпецназ', 'подаcт', 'cогласно', 'Сentral', 'ГлаваElf', 'корпуc', 'фирмыMabetex']

топ-10 (регистр как в тексте): [('Яndex', 38), ('cообщает', 7), ('МI', 7), ('Yтро', 6), ('СNN', 6), ('Cотрудники', 6), ('Аssociated', 6), ('Cтатье', 6), ('cовета', 5), ('Сentral', 4)]
топ-5 без учёта регистра: [('яndex', 38), ('cообщает', 7), ('мi', 7), ('yтро', 6), ('сnn', 6)]


In [14]:
seq = 'об этом сообщает агентство интерфакс со ссылкой на пресс-службу'.split()
tri_seq = list(zip(seq, seq[1:], seq[2:]))
bi_set = set(zip(seq, seq[1:]))
tri_set = set(tri_seq)

PUNCT_A = string.punctuation                          # только ASCII-пунктуация
PUNCT_B = string.punctuation + '«»—–…“”„‘’'           # плюс русская типографика


def lowest_trigram(punct):
    tokens = [t for t in (w.strip(punct) for w in text.lower().split()) if t]
    tri_c, bi_c = Counter(), Counter()
    for a, b, c in zip(tokens, tokens[1:], tokens[2:]):
        if (a, b) in bi_set:
            bi_c[(a, b)] += 1
            if (a, b, c) in tri_set:
                tri_c[(a, b, c)] += 1
    rows = []
    for t in tri_seq:
        denom = bi_c[t[:2]]
        if denom == 0:                                # деление на ноль, пропускаем
            continue
        rows.append((tri_c[t] / denom, tri_c[t], denom, ' '.join(t)))
    rows.sort()
    return rows


for name, punct in [('string.punctuation', PUNCT_A), ('+ русская типографика', PUNCT_B)]:
    print(f'--- strip({name})')
    for p, n3, n2, tri in lowest_trigram(punct):
        print(f'{p:.6f}  count(w1w2w3)={n3:<6} count(w1w2)={n2:<6} {tri}')

--- strip(string.punctuation)
0.020349  count(w1w2w3)=7      count(w1w2)=344    сообщает агентство интерфакс
0.074074  count(w1w2w3)=2      count(w1w2)=27     агентство интерфакс со
0.076389  count(w1w2w3)=22     count(w1w2)=288    этом сообщает агентство
0.111468  count(w1w2w3)=139    count(w1w2)=1247   ссылкой на пресс-службу
0.159910  count(w1w2w3)=284    count(w1w2)=1776   об этом сообщает
0.967742  count(w1w2w3)=180    count(w1w2)=186    интерфакс со ссылкой
0.992006  count(w1w2w3)=1241   count(w1w2)=1251   со ссылкой на
--- strip(+ русская типографика)
0.020349  count(w1w2w3)=7      count(w1w2)=344    сообщает агентство интерфакс
0.074074  count(w1w2w3)=2      count(w1w2)=27     агентство интерфакс со
0.076389  count(w1w2w3)=22     count(w1w2)=288    этом сообщает агентство
0.111468  count(w1w2w3)=139    count(w1w2)=1247   ссылкой на пресс-службу
0.159910  count(w1w2w3)=284    count(w1w2)=1776   об этом сообщает
0.967742  count(w1w2w3)=180    count(w1w2)=186    интерфакс со ссылк

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

URL = "https://github.com/mannefedov/compling_nlp_hse_course/raw/master/data/lenta_40k.csv.zip"

lenta = pd.read_csv(URL).sample(500, random_state=8).reset_index(drop=True)
print("Размер подвыборки:", lenta.shape)
print("Колонки:", list(lenta.columns))

# 2) Модель
model = SentenceTransformer('cointegrated/rubert-tiny2')

# 3) Эмбеддинги (без префиксов)
texts = lenta['text'].astype(str).tolist()
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32,
)

sim_matrix = cosine_similarity(embeddings)

np.fill_diagonal(sim_matrix, -np.inf)

i, j = np.unravel_index(np.argmax(sim_matrix), sim_matrix.shape)
if i > j:
    i, j = j, i

print(f"\nМаксимальная близость: {sim_matrix[i, j]:.4f}")
print(f"i = {i}, j = {j}")
print(f"Текст A: {texts[i][:150]}")
print(f"Текст B: {texts[j][:150]}")

first_word_a = texts[i].split()[0] if texts[i].split() else ""
first_word_b = texts[j].split()[0] if texts[j].split() else ""
print(f"\nОтвет для формы: {first_word_a} {first_word_b}")

FileNotFoundError: [Errno 2] No such file or directory: 'lenta_40k.csv.zip'